# **Regression: Multi-Layer Perceptron (MLP Regressor)**

## **Justification of Preprocessing Strategy**

### **Strict Requirement for Standardization**
The Multi-Layer Perceptron (MLP) learns by optimizing a highly non-convex loss function using Gradient Descent (specifically backpropagation). The algorithm continuously updates a massive matrix of interconnected weights based on the errors it calculates. If the input features are not on the same scale, features with larger magnitudes will produce disproportionately large gradients. This causes the optimization algorithm to bounce erratically across the error surface, either failing to converge or converging infinitely slowly. Therefore, to ensure mathematical stability and efficient learning, **Standardization (`StandardScaler`) is strictly mandatory** for all MLP models.

### **From Single-Layer to Multi-Layer (Handling Non-Linearity)**
Unlike the Single-Layer Perceptron, which is mathematically restricted to drawing straight lines (linear boundaries), the MLP introduces **Hidden Layers**. By applying non-linear activation functions (like ReLU) within these hidden layers, the network can model highly complex, multi-dimensional curves. Our hyperparameter tuning will focus heavily on discovering the optimal architecture (number of layers and neurons) and the appropriate regularization penalty (`alpha`) to prevent the network from memorizing the training data.

### **Computational Feasibility: Strategic Downsampling**
Training a neural network via iterative backpropagation is computationally expensive. When combined with rigorous cross-validation across dozens of deep learning architectures in GridSearchCV and Optuna, processing the full 100,000-row dataset risks severe hardware bottlenecks and memory exhaustion. To maintain a viable experimental timeframe while still providing the network with a vast amount of data to learn complex non-linear patterns, we applied a **strategic down-sampling to 50,000 rows**. This guarantees operational efficiency without sacrificing the integrity of the model's predictive mechanics.

## **Experiment Design & Generalization Policy**

We designed a tournament of 3 optimization levels. We explicitly enforce our **Generalization Filter**: we log both Train and Test MAE and RMSE metrics simultaneously. Any model architecture that produces a perfect training score (memorization) but fails to translate that performance to the test set will be disqualified.

* **Baseline (Strict Defaults)**: Executing the MLPRegressor with Scikit-Learn's default architecture (one hidden layer of 100 neurons, ReLU activation, Adam optimizer).
* **GridSearchCV**: A targeted 3-fold cross-validated search exploring discrete architectural steps: Shallow networks (fewer neurons) vs. Deep networks (multiple layers), alongside varying the `alpha` penalty to enforce weight decay.
* **Optuna Optimization**: Bayesian optimization deployed to dynamically explore the continuous parameter space of the network. We optimize the exact number of neurons across 1 to 3 hidden layers, the learning rate initialization, and the continuous regularization penalty, aiming to minimize validation MAE without violating our generalization criteria.

In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Ignore ConvergenceWarnings to keep the output clean during intensive Optuna/Grid searches
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_MLP")

#Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target 
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# Sub-sampling to 50,000 rows to ensure MLP converges in a reasonable timeframe for the project
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=50000, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
SEED = 42

def log_regression_metrics(y_tr_true, y_tr_pred, y_te_true, y_te_pred, duration):
    """Logs Train and Test metrics explicitly to monitor the Overfitting Gap"""
    # Train Partition Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr_true, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr_true, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr_true, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te_true, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te_true, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te_true, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)
    
# Scaling is STRICTLY MANDATORY for MLP
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# ---------------------------------------------------------
# RUN 1: BASELINE
# ---------------------------------------------------------
with mlflow.start_run(run_name="MLP_Reg_Baseline"):
    # Default: hidden_layer_sizes=(100,), activation='relu', solver='adam'
    reg_base = MLPRegressor(random_state=SEED, max_iter=500, early_stopping=True)
    
    start_time = time.time()
    reg_base.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    # Explicit Predictions
    y_pred_train_base = reg_base.predict(X_train_scaled)
    y_pred_test_base = reg_base.predict(X_test_scaled)
    
    mlflow.log_params(reg_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_regression_metrics(y_train, y_pred_train_base, y_test, y_pred_test_base, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="MLP_Reg_GridSearch"):
    param_grid = {
        "hidden_layer_sizes": [(50,), (100, 50), (50, 50, 50)], # Testing Shallow vs Deep
        "alpha": [0.0001, 0.01, 0.1], # Weight penalty
        "learning_rate_init": [0.001, 0.01]
    }

    grid_reg = GridSearchCV(
        estimator=MLPRegressor(random_state=SEED, max_iter=500, early_stopping=True),
        param_grid=param_grid,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED), # 3-fold for speed
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    start_time = time.time()
    grid_reg.fit(X_train_scaled, y_train)
    duration = time.time() - start_time

    best_mlp_grid = grid_reg.best_estimator_
    
    # Explicit Predictions
    y_pred_train_grid = best_mlp_grid.predict(X_train_scaled)
    y_pred_test_grid = best_mlp_grid.predict(X_test_scaled)

    mlflow.log_params(grid_reg.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_regression_metrics(y_train, y_pred_train_grid, y_test, y_pred_test_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective_reg(trial):
    # Dynamically building the architecture based on trial suggestions
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []
    for i in range(n_layers):
        layers.append(trial.suggest_int(f"n_units_l{i}", 20, 100))
        
    params = {
        "hidden_layer_sizes": tuple(layers),
        "alpha": trial.suggest_float("alpha", 1e-5, 1.0, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)
    }

    model = MLPRegressor(
        **params, 
        random_state=SEED, 
        max_iter=500, 
        early_stopping=True 
    )
    
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED), 
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    return -scores.mean()

with mlflow.start_run(run_name="MLP_Reg_Optuna"):
    study_reg = optuna.create_study(direction="minimize")
    
    start_time = time.time()
    study_reg.optimize(objective_reg, n_trials=10) # Safe budget for Neural Networks
    duration = time.time() - start_time

    # Reconstruct the optimal architecture
    best_params = study_reg.best_params.copy()
    n_layers_best = best_params.pop("n_layers")
    best_layers = tuple(best_params.pop(f"n_units_l{i}") for i in range(n_layers_best))
    
    best_mlp_optuna = MLPRegressor(
        hidden_layer_sizes=best_layers, 
        alpha=best_params["alpha"], 
        learning_rate_init=best_params["learning_rate_init"],
        random_state=SEED, 
        max_iter=500, 
        early_stopping=True
    )
    best_mlp_optuna.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_train_optuna = best_mlp_optuna.predict(X_train_scaled)
    y_pred_test_optuna = best_mlp_optuna.predict(X_test_scaled)

    mlflow.log_params(study_reg.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_regression_metrics(y_train, y_pred_train_optuna, y_test, y_pred_test_optuna, duration)

2026/05/22 22:32:16 INFO mlflow.tracking.fluent: Experiment with name 'Regression_MLP' does not exist. Creating a new experiment.



--- Starting Data Scaling (Standardization) ---


[I 2026-05-22 22:33:19,708] A new study created in memory with name: no-name-191e9226-5c08-44bc-a6b8-3d117f62bec1
[I 2026-05-22 22:33:23,711] Trial 0 finished with value: 0.10724786375621997 and parameters: {'n_layers': 2, 'n_units_l0': 42, 'n_units_l1': 41, 'alpha': 1.4357155903809143e-05, 'learning_rate_init': 0.0011655717143277753}. Best is trial 0 with value: 0.10724786375621997.
[I 2026-05-22 22:33:25,573] Trial 1 finished with value: 0.09805173196143187 and parameters: {'n_layers': 1, 'n_units_l0': 25, 'alpha': 0.16028958565511187, 'learning_rate_init': 0.061454738083909576}. Best is trial 1 with value: 0.09805173196143187.
[I 2026-05-22 22:33:29,421] Trial 2 finished with value: 0.09672862116344967 and parameters: {'n_layers': 3, 'n_units_l0': 20, 'n_units_l1': 89, 'n_units_l2': 84, 'alpha': 0.0015532904839138573, 'learning_rate_init': 0.04165692624806529}. Best is trial 2 with value: 0.09672862116344967.
[I 2026-05-22 22:33:33,297] Trial 3 finished with value: 0.097360582152981

## Winner Run Selection (Priority Elimination Framework)

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require Train→Test behavior to remain stable enough to indicate acceptable generalization. Runs with strong memorization patterns are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** remove runs with overfitting or underfitting.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs with catastrophic error spikes.
4. **Priority 3 (10%): Acceptable R² (Test)** — confirms explanatory quality.
5. **Tiebreaker: Lowest Fit Time** — only if MAE/RMSE/R² are effectively tied.

### Runs Summary

| Run | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| MLP_Reg_Baseline | 0.10658 | 0.11391 | 0.14727 | 0.16259 | 0.99974 | 0.99968 | 6.24s |
| MLP_Reg_GridSearch | 0.08632 | 0.08912 | 0.11489 | 0.12145 | 0.99984 | 0.99982 | 55.87s |
| MLP_Reg_Optuna | 0.07377 | 0.07429 | 0.11653 | 0.11418 | 0.99983 | 0.99984 | 43.49s |

### Generalization Check (Test − Train)
- **MLP_Reg_Baseline:** MAE gap = 0.11391 − 0.10658 = **+0.00732** and RMSE gap = 0.16259 − 0.14727 = **+0.01532** → PASS.
- **MLP_Reg_GridSearch:** MAE gap = 0.08912 − 0.08632 = **+0.00280** and RMSE gap = 0.12145 − 0.11489 = **+0.00656** → PASS.
- **MLP_Reg_Optuna:** MAE gap = 0.07429 − 0.07377 = **+0.00052** and RMSE gap = 0.11418 − 0.11653 = **−0.00235** → PASS.

### Overfitting / Underfitting Validation
- None of the runs shows overfitting. Train/Test differences are small and controlled.
- None of the runs shows underfitting. All Test R² values are extremely high.
- There is no RMSE explosion relative to MAE in any run.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: all three runs.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- MLP_Reg_Optuna: 0.07429
- MLP_Reg_GridSearch: 0.08912
- MLP_Reg_Baseline: 0.11391
- Lowest MAE: **MLP_Reg_Optuna**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- MLP_Reg_Optuna: 0.11418
- MLP_Reg_GridSearch: 0.12145
- MLP_Reg_Baseline: 0.16259
- MLP_Reg_Optuna remains the best choice.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- MLP_Reg_Optuna: 0.99984
- MLP_Reg_GridSearch: 0.99982
- MLP_Reg_Baseline: 0.99968
- MLP_Reg_Optuna also leads on R².

### Final Decision
**Winner: MLP_Reg_Optuna**

**Justification:** `MLP_Reg_Optuna` is the strongest run among those that pass the generalization filter. It has the lowest Test MAE, the lowest Test RMSE, and the highest Test R². Fit time is not needed as a tiebreaker.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **n_layers** | 3 |
| **n_units_l0** | 20 |
| **n_units_l1** | 89 |
| **n_units_l2** | 84 |
| **alpha** | 0.0015532904839138573 |
| **learning_rate_init** | 0.04165692624806529 |
| **random_state** | 42 |
| **max_iter** | 500 |
| **early_stopping** | True |
